# Final Project Preliminary Proposal

**Author:** Annie Lam

**Course:** HCDE 410

**Date:** November 2026

## Motivation & Problem Statement

Animal shelters are a community resource, and how well they're funded and staffed tends to
track the same neighborhood-level disparities seen in other public services. I want to ask:
**does an animal's outcome at an Austin city shelter — time in shelter, or whether it's
euthanized — differ depending on the ZIP code it was found in, and does that pattern track
neighborhood income?**

Practically, if certain ZIP codes consistently produce worse shelter outcomes, that's actionable
information for where to target spay/neuter outreach or low-cost vet clinics. Scientifically,
it's a human-centered case study in whether resource disparities well-documented elsewhere
(health, education) also show up somewhere easy to overlook: public animal welfare. It's the
same shape as the course's segregation/COVID-outcomes example — cross-referencing an outcomes
dataset against a geographic demographic dataset to check for a disparity. I don't have a fixed
hypothesis yet; at this stage my goal is confirming the data can support this kind of comparison,
which the exploration below does — and an early look already complicates the simple story (see
Unknowns).

## Data Used

**Primary: Austin Animal Center Intakes & Outcomes.** City of Austin's open data portal
publishes two linked datasets covering animals entering the shelter: an **Intakes** file (one
row per intake event, including a `Found Address` field with the stray's pickup location) and
an **Outcomes** file (one row per outcome, including `Outcome Status`, `Days in Shelter`, and
the matching `Intake Date`).

- Dataset pages: https://data.austintexas.gov/d/pyqf-r2dc (Intakes),
  https://data.austintexas.gov/d/gsvs-ypi7 (Outcomes)
- Direct CSV downloads: `.../api/views/{id}/rows.csv?accessType=DOWNLOAD`
- License: City of Austin Open Data Terms of Use (public reuse intended). I'll confirm the exact
  wording before publicly hosting the raw files in this repo, per the course's requirement that
  re-hosting be license-compatible.

**Secondary: U.S. Census ACS**, 5-year median household income by ZIP Code Tabulation Area
(table B19013), via data.census.gov. A U.S. government work — public domain, no restrictions.
(Not yet joined in — see Preliminary Exploration below.)

**Why this fits:** answering the question requires one dataset of geographically-located animal
outcomes and one of neighborhood income to join against it.

**Ethical considerations:** the shelter data has no owner/personal information, keeping privacy
risk low. The bigger risk is interpretive — I need to avoid framing results as "this
neighborhood doesn't take care of its animals," which would stigmatize under-resourced
communities, especially since my own exploration below already shows the relationship isn't a
simple low-income-equals-worse-outcomes story. I'll frame findings around service access,
infrastructure, and animal-type mix, not residents' choices. I'll also watch for the ecological
fallacy: ZIP-level income isn't any individual owner's income, so conclusions stay at the
neighborhood level.

## Preliminary Data Exploration

I downloaded both CSVs directly from the City of Austin portal and ran a feasibility pass before
committing to this proposal. This is exactly what I did, including a join bug I found and fixed
along the way.

**Step 1 — Load both files.**

In [ ]:
import pandas as pd
import re

intakes = pd.read_csv("data/Austin_Animal_Center_Intakes.csv")
outcomes = pd.read_csv("data/Austin_Animal_Center_Outcomes.csv")

print("Intakes:", intakes.shape)
print("Outcomes:", outcomes.shape)

**Step 2 — First join attempt, on `Animal ID` alone.**

My first instinct was to join Outcomes to Intakes on `Animal ID` to pull in the `Found Address`
field. This was wrong: 688 animals have more than one intake record (repeat strays), so joining
on ID alone matches each outcome to *every* intake that animal ever had, silently inflating the
row count.

In [ ]:
# Naive join -- kept for transparency, this is what I caught and fixed.
naive_merge = outcomes.merge(
    intakes[["Animal ID", "Found Address"]],
    on="Animal ID",
    how="left"
)
print("Outcomes before join:", len(outcomes))
print("Rows after naive ID-only join:", len(naive_merge), "<- inflated, wrong")

dupe_ids = intakes["Animal ID"].value_counts()
print("Animal IDs with more than one intake record:", (dupe_ids > 1).sum())

**Step 3 — Fixed join: match on `Animal ID` *and* the exact intake timestamp.**

Both files store a precise intake timestamp (`Source Date` in Intakes, `Intake Date` in
Outcomes), so joining on both columns together uniquely matches each outcome to the specific
intake visit it belongs to, instead of any visit by that animal.

In [ ]:
merged = outcomes.merge(
    intakes[["Animal ID", "Source Date", "Found Address"]],
    left_on=["Animal ID", "Intake Date"],
    right_on=["Animal ID", "Source Date"],
    how="left"
)

print("Outcomes before join:", len(outcomes))
print("Rows after corrected join:", len(merged), "<- matches, as it should")
print("Rows with a matched address:", merged["Found Address"].notna().sum(),
      f"({merged['Found Address'].notna().mean():.1%})")

**Step 4 — Extract ZIP code from `Found Address`.**

`Found Address` is a free-text field (e.g. `"708 Simonetti Drive, AUSTIN, TX, 78748, United
States"`), not a clean ZIP column, so I pull the 5-digit ZIP out with a regex. Addresses with no
street-level detail (just `"TX, United States"`) correctly return no match and get dropped.

In [ ]:
def extract_zip(addr):
    if pd.isna(addr):
        return None
    match = re.search(r"\b(\d{5})\b", addr)
    return match.group(1) if match else None

merged["zip"] = merged["Found Address"].apply(extract_zip)

usable = merged["zip"].notna().sum()
print(f"Rows with a usable ZIP: {usable} / {len(merged)} ({usable/len(merged):.1%})")

**Step 5 — Define euthanasia rate and look at the ZIP-level pattern.**

I flagged `Outcome Status` values of `"Euthanized"` or `"Euthanasia - RTO"` as the euthanasia
outcome, and grouped by ZIP (restricted to ZIPs with at least 50 outcomes, so small-sample noise
doesn't dominate).

In [ ]:
merged["euthanized"] = merged["Outcome Status"].isin(["Euthanized", "Euthanasia - RTO"])
print("Overall euthanasia rate:", round(merged["euthanized"].mean(), 4))

by_zip = (
    merged.dropna(subset=["zip"])
    .groupby("zip")
    .agg(n=("Animal ID", "count"),
         euth_rate=("euthanized", "mean"),
         median_days=("Days in Shelter", "median"))
    .query("n >= 50")
    .sort_values("euth_rate", ascending=False)
)

print(f"\n{len(by_zip)} ZIP codes with >= 50 outcomes")
print("\nHighest euthanasia-rate ZIPs:")
print(by_zip.head(5))
print("\nLowest euthanasia-rate ZIPs:")
print(by_zip.tail(5))

**What this shows:** the join and ZIP-extraction approach works cleanly (81% ZIP coverage
after the corrected join), and there's real variation in euthanasia rate across ZIP codes
(0.8%–27.3% among ZIPs with 50+ outcomes). The top outlier, 78701, is downtown Austin — one of
the city's *higher*-income ZIPs, not a lower-income one. That's a useful early signal: the
relationship may not be a simple "lower income → worse outcomes" story, and could instead be
driven by animal-type mix (e.g., wildlife intakes downtown) rather than neighborhood resources.
I haven't joined Census income data yet — that's next — but this confirms the pipeline is
technically sound and already surfaces something worth investigating rather than assuming.

## Unknowns & Dependencies

1. **Join correctness.** I found and fixed a real join bug above (ID-only join silently
   duplicated rows for animals with repeat intakes). I'm now joining on ID + exact intake
   timestamp, which matches the outcomes file's row count exactly — but I should still spot-check
   edge cases (e.g., same-animal intakes on the same timestamp, if any exist).
2. **ZIP coverage isn't complete.** ~19% of outcome records can't be matched to a ZIP (no
   street-level address, or no matching intake record at all). I'll need to decide whether to
   drop these or analyze them separately as "location unknown."
3. **License confirmation.** I'm treating the City of Austin's data as reuse-friendly based on
   it being an open government portal, but want to confirm the exact terms before publicly
   hosting the raw files in this repo.
4. **Confounding variables.** The downtown ZIP finding above suggests animal type (e.g.,
   wildlife vs. domestic pets) may matter as much as or more than income — I'll need to control
   for this rather than attribute everything to neighborhood resources.
5. **Census join not yet done.** I've confirmed the shelter-side pipeline works; joining in ACS
   income-by-ZIP data is the next step, not yet attempted.